In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("catalogo", "maintenance_iot")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("source_table", "sensor_events")
dbutils.widgets.text("target_table", "sensor_events_clean")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")

# ruta = f"abfss://{container}@{datalake}.dfs.core.windows.net/circuits.csv"

PASO 2 — Leer Bronze

In [0]:
df_bronze = spark.table(f"{catalogo}.{bronze_schema}.{source_table}")

print("Registros en Bronze:", df_bronze.count())

🔹 PASO 3 — Limpieza básica

Eliminamos valores nulos

Filtramos valores negativos (si no aplican)


In [0]:
df_clean = (
    df_bronze
    .filter("value IS NOT NULL")
    .filter("value >= 0")
)

🔹 PASO 4 — Ventana para cálculos

Queremos calcular métricas por:

rig_id

sensor_id

ordenado por tiempo


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, stddev, col

window_spec = Window \
    .partitionBy("rig_id", "sensor_id") \
    .orderBy("event_timestamp") \
    .rowsBetween(-10, 0)

🔹 PASO 5 — Calcular métricas

In [0]:
df_metrics = (
    df_clean
    .withColumn("moving_avg", avg("value").over(window_spec))
    .withColumn("std_dev", stddev("value").over(window_spec))
)

🔹 PASO 6 — Detectar anomalías

Regla simple:

Si valor > media + 3*std → anomaly

In [0]:
from pyspark.sql.functions import when

df_silver = (
    df_metrics
    .withColumn(
        "anomaly_flag",
        when(
            col("value") > col("moving_avg") + 3 * col("std_dev"),
            "ANOMALY"
        ).otherwise("NORMAL")
    )
)

In [0]:
df_final = df_silver.select(
    "event_id",
    "rig_id",
    "sensor_id",
    "sensor_type",
    "value",
    "unit",
    "event_timestamp",
    "moving_avg",
    "std_dev",
    "anomaly_flag",
    "ingestion_date"
)

PASO 8 — Insertar en Silver

In [0]:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{silver_schema}.{target_table}")

In [0]:
spark.sql(f"""
SELECT anomaly_flag, COUNT(*)
FROM {catalogo}.{silver_schema}.{target_table}
GROUP BY anomaly_flag
""").show()